In [3]:
!pip install torch torchvision h5py scikit-image Pillow

In [13]:
!python eval.py --model data/FC/fc-model.pth --infos_path data/FC/fc-infos.pkl --image_folder data/ --num_images 1

In [46]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import json
import urllib.request
import os
import glob

def initialize_model():
    """
    Initializes the Swin Transformer V2 model with pre-trained weights
    and fetches the ImageNet class index mapping.
    """
    print("Loading pre-trained Swin Transformer V2 model...")
    model = models.swin_v2_b(weights=models.Swin_V2_B_Weights.DEFAULT)
    model.eval()
    
    print("Fetching ImageNet class index labels...\n")
    labels_url = "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json"
    with urllib.request.urlopen(labels_url) as url:
        class_idx = json.loads(url.read().decode())
        
    return model, class_idx

def classify_image_batch(data_dir, model, class_idx):
    """
    Scans the specified directory, processes all valid images,
    and executes inference, displaying results in a clean columnar format.
    """
    supported_extensions = ['*.jpg', '*.jpeg', '*.png', '*.webp']
    image_paths = []
    
    for ext in supported_extensions:
        image_paths.extend(glob.glob(os.path.join(data_dir, ext)))
        
    if not image_paths:
        print(f"No valid image files found in directory: '{data_dir}'")
        return

    transform = transforms.Compose([
        transforms.Resize(260, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print(f"Found {len(image_paths)} images. Commencing inference batch...\n")
    
    # Cabeçalho das colunas bem estruturado
    # Formato: Esquerda (25 caracteres) | Esquerda (25 caracteres) | Direita (12 caracteres)
    print(f"{'TARGET FILE':<25} | {'DETECTED OBJECT':<25} | {'CONFIDENCE':>12}")
    print("-" * 68)

    for path in image_paths:
        try:
            image = Image.open(path).convert('RGB')
            image_tensor = transform(image).unsqueeze(0)
            
            with torch.no_grad():
                outputs = model(image_tensor)
                probabilities = torch.nn.functional.softmax(outputs, dim=1)[0] * 100
            
            _, indices = torch.sort(outputs, descending=True)
            top3_indices = indices[0][:3]
            
            file_name = os.path.basename(path)
            
            # Para cada imagem, exibe os 3 principais resultados alinhados em colunas
            for idx in top3_indices:
                class_id = idx.item()
                label = class_idx[str(class_id)][1]
                confidence = probabilities[class_id].item()
                
                formatted_label = label.replace('_', ' ').title()
                
                # Imprime os dados respeitando o espaçamento das colunas
                print(f"{file_name:<25} | {formatted_label:<25} | {confidence:>11.2f}%")
                
                # Deixa o nome do arquivo em branco nas linhas seguintes do Top 3 para não poluir
                file_name = ""
                
            print("-" * 68) # Linha sutil para separar uma imagem da outra
            
        except Exception as e:
            print(f"Failed to process file {os.path.basename(path)}. Error: {e}")

if __name__ == "__main__":
    model_instance, labels_mapping = initialize_model()
    classify_image_batch('data', model_instance, labels_mapping)

Loading pre-trained Swin Transformer V2 model...
Fetching ImageNet class index labels...

Found 3 images. Commencing inference batch...

TARGET FILE               | DETECTED OBJECT           |   CONFIDENCE
--------------------------------------------------------------------
ferrari.jpg               | Racer                     |       92.20%
                          | Sports Car                |        0.79%
                          | Car Wheel                 |        0.61%
--------------------------------------------------------------------
lion.jpg                  | Lion                      |       90.65%
                          | Cougar                    |        0.10%
                          | Hyena                     |        0.04%
--------------------------------------------------------------------
shark.jpg                 | Great White Shark         |       88.79%
                          | Tiger Shark               |        0.69%
                          | Killer 